In [1]:
# %% [markdown]
# # 10 — Pricing Adequacy Analysis
#
# ## Objective
# Evaluate pricing adequacy across the SecureHealth portfolio.
#
# This notebook will:
# - combine premium and observed cost information
# - estimate pricing adequacy
# - classify policies as underpriced / fair / overpriced
# - generate a first suggested premium range
#
# Main output:
# - `policy_pricing_review.csv`

# %%
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

# %%
import pandas as pd
import numpy as np

from src.config import PROCESSED_DIR, OUTPUTS_DIR

# %% [markdown]
# ## Load bases

# %%
member_master = pd.read_csv(PROCESSED_DIR / "member_master.csv")
claims_analytical_base = pd.read_csv(PROCESSED_DIR / "claims_analytical_base.csv")

risk_scores_path = OUTPUTS_DIR / "member_risk_scores.csv"
member_risk_scores = pd.read_csv(risk_scores_path) if risk_scores_path.exists() else None

print("member_master:", member_master.shape)
print("claims_analytical_base:", claims_analytical_base.shape)
print("member_risk_scores:", None if member_risk_scores is None else member_risk_scores.shape)

# %%
display(member_master.head(3))

# %% [markdown]
# ## Detect claim cost column

# %%
claim_cost_col = None
for candidate in ["claim_amount", "paid_amount", "approved_amount", "amount_paid", "cost_amount", "total_cost"]:
    if candidate in claims_analytical_base.columns:
        claim_cost_col = candidate
        break

print("Detected claim cost column:", claim_cost_col)

# %% [markdown]
# ## Build member/policy observed cost base

# %%
pricing_base = member_master.copy()

if "member_id" in claims_analytical_base.columns:
    if claim_cost_col is not None:
        member_costs = (
            claims_analytical_base.groupby("member_id")[claim_cost_col]
            .agg(["count", "sum", "mean", "median", "max"])
            .reset_index()
        )
        member_costs.columns = [
            "member_id",
            "claims_count",
            "observed_cost_sum",
            "observed_cost_mean",
            "observed_cost_median",
            "observed_cost_max",
        ]
    else:
        member_costs = (
            claims_analytical_base.groupby("member_id")
            .size()
            .reset_index(name="claims_count")
        )
        member_costs["observed_cost_sum"] = 0
        member_costs["observed_cost_mean"] = 0
        member_costs["observed_cost_median"] = 0
        member_costs["observed_cost_max"] = 0

    pricing_base = pricing_base.merge(member_costs, on="member_id", how="left")

for col in ["claims_count", "observed_cost_sum", "observed_cost_mean", "observed_cost_median", "observed_cost_max"]:
    if col in pricing_base.columns:
        pricing_base[col] = pricing_base[col].fillna(0)

if member_risk_scores is not None:
    pricing_base = pricing_base.merge(
        member_risk_scores[["member_id", "predicted_risk_probability", "predicted_risk_segment"]],
        on="member_id",
        how="left"
    )

print("pricing_base:", pricing_base.shape)
display(pricing_base.head(3))

# %% [markdown]
# ## Create pricing metrics

# %%
# clean numeric columns
for col in [
    "premium_monthly",
    "premium_annual",
    "underwriting_load_factor",
    "discount_factor",
    "pricing_adequacy_ratio",
    "observed_cost_sum",
    "observed_cost_mean",
    "predicted_risk_probability",
]:
    if col in pricing_base.columns:
        pricing_base[col] = pd.to_numeric(pricing_base[col], errors="coerce")

# expected cost proxy
if "predicted_risk_probability" in pricing_base.columns:
    portfolio_mean_cost = pricing_base["observed_cost_sum"].mean()
    pricing_base["expected_annual_cost_proxy"] = (
        pricing_base["predicted_risk_probability"].fillna(pricing_base["predicted_risk_probability"].median())
        * portfolio_mean_cost
    )
else:
    pricing_base["expected_annual_cost_proxy"] = pricing_base["observed_cost_sum"]

# pure premium proxy
pricing_base["pure_premium_proxy"] = pricing_base["expected_annual_cost_proxy"]

# loaded technical premium
if "underwriting_load_factor" in pricing_base.columns:
    pricing_base["underwriting_load_factor"] = pricing_base["underwriting_load_factor"].fillna(1.0)
else:
    pricing_base["underwriting_load_factor"] = 1.0

pricing_base["technical_premium_proxy"] = (
    pricing_base["pure_premium_proxy"] * pricing_base["underwriting_load_factor"]
)

# apply discount if present
if "discount_factor" in pricing_base.columns:
    pricing_base["discount_factor"] = pricing_base["discount_factor"].fillna(1.0)
else:
    pricing_base["discount_factor"] = 1.0

pricing_base["discounted_technical_premium_proxy"] = (
    pricing_base["technical_premium_proxy"] * pricing_base["discount_factor"]
)

# loss ratio and adequacy
pricing_base["observed_loss_ratio"] = np.where(
    pricing_base["premium_annual"] > 0,
    pricing_base["observed_cost_sum"] / pricing_base["premium_annual"],
    np.nan
)

pricing_base["expected_loss_ratio_proxy"] = np.where(
    pricing_base["premium_annual"] > 0,
    pricing_base["expected_annual_cost_proxy"] / pricing_base["premium_annual"],
    np.nan
)

pricing_base["premium_gap_vs_expected"] = pricing_base["premium_annual"] - pricing_base["expected_annual_cost_proxy"]
pricing_base["premium_gap_vs_technical"] = pricing_base["premium_annual"] - pricing_base["discounted_technical_premium_proxy"]

# %%
pricing_base[[
    c for c in [
        "member_id",
        "policy_id",
        "premium_annual",
        "observed_cost_sum",
        "expected_annual_cost_proxy",
        "technical_premium_proxy",
        "discounted_technical_premium_proxy",
        "observed_loss_ratio",
        "expected_loss_ratio_proxy",
    ] if c in pricing_base.columns
]].head(10)

# %% [markdown]
# ## Pricing status classification

# %%
def classify_pricing(expected_lr):
    if pd.isna(expected_lr):
        return "unknown"
    if expected_lr > 1.10:
        return "underpriced"
    elif expected_lr < 0.70:
        return "overpriced"
    else:
        return "fair"

pricing_base["pricing_status"] = pricing_base["expected_loss_ratio_proxy"].apply(classify_pricing)

# suggested premium range
pricing_base["suggested_premium_low"] = pricing_base["discounted_technical_premium_proxy"] * 0.95
pricing_base["suggested_premium_mid"] = pricing_base["discounted_technical_premium_proxy"]
pricing_base["suggested_premium_high"] = pricing_base["discounted_technical_premium_proxy"] * 1.10

# %%
pricing_base["pricing_status"].value_counts(dropna=False).rename_axis("pricing_status").reset_index(name="count")

# %% [markdown]
# ## Segment views

# %%
if "plan_type" in pricing_base.columns:
    display(
        pricing_base.groupby("plan_type")[[
            "premium_annual",
            "observed_cost_sum",
            "expected_annual_cost_proxy",
            "observed_loss_ratio",
            "expected_loss_ratio_proxy",
        ]]
        .mean()
        .reset_index()
        .sort_values("expected_loss_ratio_proxy", ascending=False)
    )

if "archetype" in pricing_base.columns:
    display(
        pricing_base.groupby("archetype")[[
            "premium_annual",
            "observed_cost_sum",
            "expected_annual_cost_proxy",
            "observed_loss_ratio",
            "expected_loss_ratio_proxy",
        ]]
        .mean()
        .reset_index()
        .sort_values("expected_loss_ratio_proxy", ascending=False)
    )

# %%
if "predicted_risk_segment" in pricing_base.columns:
    display(
        pricing_base.groupby("predicted_risk_segment")[[
            "premium_annual",
            "observed_cost_sum",
            "expected_annual_cost_proxy",
            "expected_loss_ratio_proxy",
        ]]
        .mean()
        .reset_index()
    )

# %% [markdown]
# ## Build final output

# %%
output_cols = [c for c in [
    "member_id",
    "policy_id",
    "age",
    "sex",
    "region",
    "archetype",
    "plan_type",
    "plan_tier",
    "coverage_scope",
    "provider_network_type",
    "premium_monthly",
    "premium_annual",
    "underwriting_load_factor",
    "discount_factor",
    "pricing_adequacy_ratio",
    "claims_count",
    "observed_cost_sum",
    "expected_annual_cost_proxy",
    "technical_premium_proxy",
    "discounted_technical_premium_proxy",
    "observed_loss_ratio",
    "expected_loss_ratio_proxy",
    "premium_gap_vs_expected",
    "premium_gap_vs_technical",
    "pricing_status",
    "suggested_premium_low",
    "suggested_premium_mid",
    "suggested_premium_high",
    "predicted_risk_probability",
    "predicted_risk_segment",
    "renewal_flag",
    "cancellation_flag",
] if c in pricing_base.columns]

policy_pricing_review = pricing_base[output_cols].copy()
print("policy_pricing_review:", policy_pricing_review.shape)
display(policy_pricing_review.head(10))

# %% [markdown]
# ## Save output

# %%
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUTS_DIR / "policy_pricing_review.csv"
policy_pricing_review.to_csv(output_file, index=False)

print("Saved:", output_file)

# %% [markdown]
# ## Final notes

# %%
notes = [
    "This is a first pricing adequacy layer, based on observed and proxy-expected cost.",
    "The premium guidance is simplified but useful for portfolio analytics and dashboard simulation.",
    "Next step: build the prospect profiler."
]

for i, note in enumerate(notes, 1):
    print(f"{i}. {note}")

PROJECT_ROOT: C:\Users\dolivares\Desktop\novares-securehealth
member_master: (4000, 63)
claims_analytical_base: (44144, 107)
member_risk_scores: (4000, 17)


,member_id,policy_id,join_date,age,age_band,sex,region,city_tier,socioeconomic_band,employment_status,...,sales_channel,broker_id,tenure_days,tenure_months_approx,premium_gap,has_dependents_flag,high_chronic_burden_flag,high_baseline_risk_flag,high_utilization_propensity_flag,high_misuse_exposure_flag
0,MBR000001,POL000001,2022-09-24,47,45-54,M,San Pedro,Urban,Upper-Middle,Self-Employed,...,Corporate,NaN,1559,51.3,4.547474e-13,0,0,1,1,1
1,MBR000002,POL000002,2024-04-02,35,35-44,F,San Pedro,Metro,Lower-Middle,Employed,...,Direct,NaN,1003,33.0,4.547474e-13,0,0,0,0,0
2,MBR000003,POL000003,2024-06-27,65,65+,M,Asunción,Urban,Middle,Self-Employed,...,Direct,NaN,917,30.2,-9.094947e-13,1,0,1,1,0


Detected claim cost column: None
pricing_base: (4000, 70)


,member_id,policy_id,join_date,age,age_band,sex,region,city_tier,socioeconomic_band,employment_status,...,high_baseline_risk_flag,high_utilization_propensity_flag,high_misuse_exposure_flag,claims_count,observed_cost_sum,observed_cost_mean,observed_cost_median,observed_cost_max,predicted_risk_probability,predicted_risk_segment
0,MBR000001,POL000001,2022-09-24,47,45-54,M,San Pedro,Urban,Upper-Middle,Self-Employed,...,1,1,1,49.0,0.0,0.0,0.0,0.0,0.997812,very_high
1,MBR000002,POL000002,2024-04-02,35,35-44,F,San Pedro,Metro,Lower-Middle,Employed,...,0,0,0,1.0,0.0,0.0,0.0,0.0,0.000000,very_low
2,MBR000003,POL000003,2024-06-27,65,65+,M,Asunción,Urban,Middle,Self-Employed,...,1,1,0,24.0,0.0,0.0,0.0,0.0,0.979234,very_high


,plan_type,premium_annual,observed_cost_sum,expected_annual_cost_proxy,observed_loss_ratio,expected_loss_ratio_proxy
0,Chronic Care,7107.271356,0.0,0.0,0.0,0.0
1,Essential,2053.783097,0.0,0.0,0.0,0.0
2,Family Plus,5957.219416,0.0,0.0,0.0,0.0
3,High Protection,8320.270654,0.0,0.0,0.0,0.0
4,Managed Review,4462.956491,0.0,0.0,0.0,0.0
5,Standard,3608.234464,0.0,0.0,0.0,0.0


,archetype,premium_annual,observed_cost_sum,expected_annual_cost_proxy,observed_loss_ratio,expected_loss_ratio_proxy
0,Chronic Managed,7107.271356,0.0,0.0,0.0,0.0
1,Family Planner,5957.219416,0.0,0.0,0.0,0.0
2,Healthy Low User,2053.783097,0.0,0.0,0.0,0.0
3,High Acute Risk,8320.270654,0.0,0.0,0.0,0.0
4,Hyper-Utilizer / Misuse Risk,4462.956491,0.0,0.0,0.0,0.0
5,Preventive User,3608.234464,0.0,0.0,0.0,0.0


,predicted_risk_segment,premium_annual,observed_cost_sum,expected_annual_cost_proxy,expected_loss_ratio_proxy
0,high,7362.260000,0.0,0.0,0.0
1,medium,6718.760000,0.0,0.0,0.0
2,very_high,6559.423857,0.0,0.0,0.0
3,very_low,4029.375072,0.0,0.0,0.0


policy_pricing_review: (4000, 32)


,member_id,policy_id,age,sex,region,archetype,plan_type,plan_tier,coverage_scope,provider_network_type,...,premium_gap_vs_expected,premium_gap_vs_technical,pricing_status,suggested_premium_low,suggested_premium_mid,suggested_premium_high,predicted_risk_probability,predicted_risk_segment,renewal_flag,cancellation_flag
0,MBR000001,POL000001,47,M,San Pedro,Hyper-Utilizer / Misuse Risk,Managed Review,Mid,Full,Balanced,...,3496.80,3496.80,overpriced,0.0,0.0,0.0,0.997812,very_high,1,0
1,MBR000002,POL000002,35,F,San Pedro,Healthy Low User,Essential,Basic,Ambulatory,Narrow,...,2384.40,2384.40,overpriced,0.0,0.0,0.0,0.000000,very_low,1,0
2,MBR000003,POL000003,65,M,Asunción,Chronic Managed,Chronic Care,Premium,Full,Broad,...,5161.20,5161.20,overpriced,0.0,0.0,0.0,0.979234,very_high,0,0
3,MBR000004,POL000004,38,F,Alto Paraná,Healthy Low User,Essential,Basic,Ambulatory,Balanced,...,1913.52,1913.52,overpriced,0.0,0.0,0.0,0.000000,very_low,1,0
4,MBR000005,POL000005,47,M,Central,Chronic Managed,Chronic Care,Premium,Full,Balanced,...,6181.56,6181.56,overpriced,0.0,0.0,0.0,0.712598,high,1,0
5,MBR000006,POL000006,80,F,Itapúa,Chronic Managed,Chronic Care,Premium,Full,Broad,...,7655.88,7655.88,overpriced,0.0,0.0,0.0,0.956795,very_high,1,0
6,MBR000007,POL000007,30,M,Asunción,Preventive User,Standard,Mid,Full,Balanced,...,4108.08,4108.08,overpriced,0.0,0.0,0.0,0.000000,very_low,1,0
7,MBR000008,POL000008,42,F,Central,Preventive User,Standard,Mid,Full,Balanced,...,3302.52,3302.52,overpriced,0.0,0.0,0.0,0.000000,very_low,1,0
8,MBR000009,POL000009,52,M,Alto Paraná,Chronic Managed,Chronic Care,Premium,Full,Broad,...,8455.32,8455.32,overpriced,0.0,0.0,0.0,0.769704,high,1,0
9,MBR000010,POL000010,47,M,Central,Preventive User,Standard,Mid,Full,Balanced,...,3699.24,3699.24,overpriced,0.0,0.0,0.0,0.000000,very_low,1,0


Saved: C:\Users\dolivares\Desktop\novares-securehealth\data\outputs\policy_pricing_review.csv
1. This is a first pricing adequacy layer, based on observed and proxy-expected cost.
2. The premium guidance is simplified but useful for portfolio analytics and dashboard simulation.
3. Next step: build the prospect profiler.
